In [2]:
import pandas as pd
from pathlib import Path

telemetry_path = Path("data/telemetry.csv")

def load_telemetry(path=telemetry_path, timestamp_col="timestamp", index=True, chunksize=None, dtype=None, tz=None):
  """
  Load telemetry CSV into a DataFrame.
  - path: file path or Path
  - timestamp_col: name of the timestamp column to parse and optionally set as index
  - index: if True, set timestamp_col as index
  - chunksize: if provided, read in chunks and concat (useful for very large files)
  - dtype: optional dict of column dtypes for memory/perf
  - tz: optional timezone string to localize/convert the index
  """
  parse_dates = [timestamp_col] if timestamp_col else None

  if chunksize:
    it = pd.read_csv(path, parse_dates=parse_dates, infer_datetime_format=True, dtype=dtype, chunksize=chunksize)
    df = pd.concat(it)
  else:
    df = pd.read_csv(path, parse_dates=parse_dates, infer_datetime_format=True, dtype=dtype)

  if timestamp_col and index and timestamp_col in df.columns:
    df.set_index(timestamp_col, inplace=True)
    if tz:
      if getattr(df.index, "tz", None) is None:
        df.index = df.index.tz_localize(tz)
      else:
        df.index = df.index.tz_convert(tz)

  return df

# Example usage:
# df = load_telemetry("data/telemetry.csv", timestamp_col="ts", dtype={"device_id": "category"}, chunksize=100_000, tz="UTC")
# df.head()